In [7]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# [부록 1] PFAS-Free 5종 약액 가상 데이터셋 자동 생성 및 물리 엔진 시뮬레이션 코드
# (※ 김민준 AI 제어로직 변수 공간[6개 차원] 및 소논문 데이터 개수 완벽 동기화 버전)
# ==============================================================================

# 1. 초기 설정 및 시드 고정 (공학적 재현성 확보)
TOTAL_TARGET_SAMPLES = 20000
NUM_CHEMS = 5
# 💡 [팩트 동기화 1]: 소논문 명시 스펙(총 20,000개) 충족을 위해 약액당 4,000개씩 균등 분할 생성
NUM_SAMPLES_PER_CHEM = TOTAL_TARGET_SAMPLES // NUM_CHEMS
np.random.seed(42)

# 2. 친환경 대체 약액 5종 물성치 데이터베이스
chemical_db = {
    'Novec 4200':   {'gamma_base': 17.0, 'theta': 115.0, 'viscosity_base': 1.02},
    'Triton BG-10': {'gamma_base': 28.0, 'theta': 75.0,  'viscosity_base': 1.15},
    'Triton CG-50': {'gamma_base': 27.0, 'theta': 78.0,  'viscosity_base': 1.20},
    'Brij S100':    {'gamma_base': 56.0, 'theta': 65.0,  'viscosity_base': 1.50},
    'DI-03':        {'gamma_base': 72.0, 'theta': 70.0,  'viscosity_base': 1.00}  # 오존수 명칭 통일
}

# 3. 반도체 패턴 구조 및 설비 고정 상수 세팅 (설계 상수로 고정락 완비)
d_pattern = 20e-9       # 20nm 노드 공정 고정 규격 (Pitch)
k_constant = 1.2e-7     # 최신 저진동 스핀 설비 현실화 스펙
vib_constant = 1.0e-8   # 미세 진동 패널티 상수
r_max = 0.15            # 12인치 웨이퍼 최대 반경 150mm (0.15m)

print("=========================================================")
print("데이터셋 자동 생성 및 붕괴(Collapse) 통계 요약 리포트 (6변수 최종본)")
print("=========================================================")

master_df_list = []

# 4. 약액별 순회하며 데이터셋 생성
for chem_name, props in chemical_db.items():
    gamma_base = props['gamma_base']
    theta = props['theta']
    viscosity_base = props['viscosity_base']

    # 가. 독립 변수 생성 (약액별 열역학적 공정 제한 온도 반영)
    # 💡 [팩트 동기화 2]: 'DI-03' 문자열 오타 정밀 교정 완료하여 40도 제한 정상 작동 보장
    if chem_name == 'DI-03':
        temp = np.random.uniform(20, 40, NUM_SAMPLES_PER_CHEM)       # 오존 기화 방지 상한 40도
    elif chem_name == 'Brij S100':
        temp = np.random.uniform(55, 90, NUM_SAMPLES_PER_CHEM)       # 녹는점 고려한 하한 55도 제한
    else:
        temp = np.random.uniform(20, 90, NUM_SAMPLES_PER_CHEM)       # 타 약액 상온~90도 운전 마진

    # 💡 [팩트 동기화 3]: 설비 안정성을 고려한 실전형 6,000 RPM 한계 상한 엄격 적용
    rpm = np.random.uniform(500, 6000, NUM_SAMPLES_PER_CHEM)
    radius = np.random.uniform(0, r_max, NUM_SAMPLES_PER_CHEM)       # 0~150mm 공간 가변 변수

    # 나. 물리 엔진 연산: 유효 표면장력 및 안드레이드 점도 패널티
    gamma_eff = gamma_base - 0.1 * (temp - 25.0)
    phi_viscosity = 1.02 * np.exp(100 / (temp + 273.15))

    # 다. 모세관 압력 (단위: MPa)
    p_cap_pa = np.abs((2 * (gamma_eff * 1e-3) * np.cos(np.radians(theta))) / d_pattern)
    p_cap = p_cap_pa * phi_viscosity * 1e-6

    # 라. 동적 원심 압력 및 기계적 진동 패널티 (현실화 상수 적용)
    p_rpm = (radius / r_max) * (k_constant * (rpm**2))
    p_vibration = (radius / r_max) * (vib_constant * (rpm**2))

    # 마. 총 합산 응력 및 15% 가우시안 노이즈 주입
    p_total = p_cap + p_rpm + p_vibration
    damage_factor = np.random.normal(1.0, 0.15, NUM_SAMPLES_PER_CHEM)
    p_final = p_total * damage_factor

    # 구조적 임계 강도 5.0 MPa 기준 붕괴 여부 판정 (1: 붕괴, 0: 안전)
    collapse = np.where(p_final > 5.0, 1, 0)

    # 바. 데이터프레임 빌드 (김민준 ML 알고리즘 입력 피처 6개와 완벽 동기화)
    df = pd.DataFrame({
        'Temp': temp,
        'RPM': rpm,
        'Gamma': np.full(NUM_SAMPLES_PER_CHEM, gamma_base),
        'Theta': np.full(NUM_SAMPLES_PER_CHEM, theta),
        'Viscosity': np.full(NUM_SAMPLES_PER_CHEM, viscosity_base),
        'Radius': radius * 1000, # MATLAB 시뮬레이션 코드 호환을 위한 mm 단위 변환 저장
        'Collapse': collapse
    })

    # 개별 CSV 파일 저장
    save_name = chem_name.replace(" ", "_")
    df.to_csv(f'synthetic_data_corrected_{save_name}.csv', index=False)
    master_df_list.append(df)

    # 붕괴 통계 계산
    collapse_count = np.sum(collapse)
    collapse_ratio = (collapse_count / NUM_SAMPLES_PER_CHEM) * 100
    print(f"▶ {chem_name} 데이터셋 ({NUM_SAMPLES_PER_CHEM}개) 생성 완료")
    print(f"   - 붕괴 데이터: {collapse_count}개 ({collapse_ratio:.1f}%) / 정상 데이터: {NUM_SAMPLES_PER_CHEM - collapse_count}개")

# 💡 [팩트 동기화 4]: 5종 약액 데이터를 하나로 통합한 유니버설 AI 학습용 마스터 파일 강제 출력
unified_dataset = pd.concat(master_df_list, ignore_index=True)
unified_dataset.to_csv('synthetic_data_master_unified.csv', index=False)

print("=========================================================")
print(f"✔ [검증 완료] 5종 약액 데이터 총 {len(unified_dataset)}건 통합 완료 (소논문 싱크 100%)")
print("✔ 'synthetic_data_master_unified.csv' 마스터 파일 추출 완료.")
print("=========================================================")

데이터셋 자동 생성 및 붕괴(Collapse) 통계 요약 리포트 (6변수 최종본)
▶ Novec 4200 데이터셋 (4000개) 생성 완료
   - 붕괴 데이터: 32개 (0.8%) / 정상 데이터: 3968개
▶ Triton BG-10 데이터셋 (4000개) 생성 완료
   - 붕괴 데이터: 43개 (1.1%) / 정상 데이터: 3957개
▶ Triton CG-50 데이터셋 (4000개) 생성 완료
   - 붕괴 데이터: 27개 (0.7%) / 정상 데이터: 3973개
▶ Brij S100 데이터셋 (4000개) 생성 완료
   - 붕괴 데이터: 560개 (14.0%) / 정상 데이터: 3440개
▶ DI-03 데이터셋 (4000개) 생성 완료
   - 붕괴 데이터: 881개 (22.0%) / 정상 데이터: 3119개
✔ [검증 완료] 5종 약액 데이터 총 20000건 통합 완료 (소논문 싱크 100%)
✔ 'synthetic_data_master_unified.csv' 마스터 파일 추출 완료.
